<a href="https://colab.research.google.com/github/Jesus-0655/Entregable_3_Aprendizaje_Entrenamiento_Dataset/blob/main/Modelo_Entrenado_Baterias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Pipeline de datos y estructura del repositorio

In [ ]:
import importlib, subprocess, sys
for pkg in ["xgboost", "shap"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

from pathlib import Path
for d in ["src/data", "src/models", "data/processed", "models", "notebooks"]:
    Path(d).mkdir(parents=True, exist_ok=True)
for f in ["src/__init__.py", "src/data/__init__.py", "src/models/__init__.py"]:
    Path(f).touch()
print("Estructura de carpetas lista (rutas relativas al directorio de trabajo).")

In [ ]:
%%writefile requirements.txt
pandas>=2.0
numpy>=1.24
scikit-learn>=1.3
xgboost>=2.0
shap>=0.44
joblib>=1.3
matplotlib>=3.7
seaborn>=0.12

In [ ]:
%%writefile src/data/preprocess.py
"""Preprocesamiento: construccion de etiquetas, seleccion de variables,
particion Train/Test y transformador (escalado + one-hot).

Nota: el dataset ya llega con los nulos imputados con la media (paso cumplido
previamente), por eso aqui solo se verifica que no existan NaN.
"""
from __future__ import annotations

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --- Definicion de la variable objetivo (3 clases) ---------------------------
COL_FALLA = "battery_failure"
COL_VIDA = "predicted_remaining_life_cycles"
UMBRAL_MANTENIMIENTO = 6000  # ciclos de vida restante; ajustable
CLASES = {0: "Óptimo", 1: "Mantenimiento preventivo", 2: "Descarte"}

# --- Columnas que NO entran al modelo ---------------------------------------
COLS_IDENTIFICACION = [
    "vehicle_id", "vehicle_brand", "vehicle_model", "vehicle_type",
    "battery_serial", "manufacturing_year",
]
# Ademas se excluyen: todas las columnas *_score, COL_VIDA y COL_FALLA
# (con ellas se construye la etiqueta o son indices derivados -> data leakage).

SEED = 42
TEST_SIZE = 0.20


def verificar_nulos(df: pd.DataFrame) -> int:
    """El dataset ya fue imputado con la media; solo se valida."""
    n = int(df.isna().sum().sum())
    if n:
        raise ValueError(f"El dataset contiene {n} nulos; debe venir imputado.")
    return n


def construir_etiquetas(df: pd.DataFrame) -> pd.Series:
    """0 = Óptimo, 1 = Mantenimiento preventivo, 2 = Descarte.

    Descarte      : battery_failure == 1
    Mantenimiento : sin falla y vida restante < UMBRAL_MANTENIMIENTO
    Óptimo        : el resto
    """
    y = pd.Series(0, index=df.index, name="estado_bateria", dtype="int64")
    y[df[COL_VIDA] < UMBRAL_MANTENIMIENTO] = 1
    y[df[COL_FALLA] == 1] = 2
    return y


def columnas_a_excluir(df: pd.DataFrame) -> list:
    return [
        c for c in df.columns
        if c in COLS_IDENTIFICACION or c.endswith("_score") or c in (COL_FALLA, COL_VIDA)
    ]


def preparar_datos(df: pd.DataFrame):
    y = construir_etiquetas(df)
    X = df.drop(columns=columnas_a_excluir(df))
    return X, y


def dividir(X: pd.DataFrame, y: pd.Series):
    """Particion estratificada 80/20 con semilla fija (reproducible)."""
    return train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
    )


def construir_preprocesador(X: pd.DataFrame) -> ColumnTransformer:
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    num = [c for c in X.columns if c not in cat]
    return ColumnTransformer(
        [
            ("num", StandardScaler(), num),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
        ],
        verbose_feature_names_out=False,
    )

### Entrenamiento comparativo, JSON y las variables

In [ ]:
%%writefile src/models/train.py
"""Entrenamiento comparativo: XGBoost (principal) vs Regresion Logistica (baseline).

Uso:
    python src/models/train.py --data data/processed/baterias_procesado.csv --output models/
"""
from __future__ import annotations

import argparse
import json
import sys
import time
from datetime import datetime
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             classification_report, f1_score, recall_score)
from sklearn.model_selection import (GridSearchCV, ParameterSampler,
                                     StratifiedKFold, train_test_split)
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

from src.data import preprocess as pp

LABELS = [0, 1, 2]
NOMBRES = [pp.CLASES[i] for i in LABELS]


def log(msg: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)


def pesos(y, potencia: float):
    """Pesos por muestra: (balanced) ** potencia. 0 = sin pesos, 1 = balanced."""
    return compute_sample_weight("balanced", y) ** potencia


def diagnostico(f1_train: float, f1_test: float) -> str:
    brecha = f1_train - f1_test
    if f1_train < 0.60:
        return "Underfitting (bajo rendimiento incluso en Train)"
    if brecha > 0.10:
        return f"Overfitting (brecha F1 = {brecha:.3f})"
    if brecha > 0.05:
        return f"Sobreajuste moderado (brecha F1 = {brecha:.3f})"
    return f"Ajuste adecuado (brecha F1 = {brecha:.3f})"


def evaluar(nombre, modelo, Xtr, ytr, Xte, yte, t_fit, hiperparametros):
    t0 = time.perf_counter()
    p_te = modelo.predict(Xte)
    t_pred = time.perf_counter() - t0
    p_tr = modelo.predict(Xtr)
    rec = recall_score(yte, p_te, labels=LABELS, average=None, zero_division=0)
    m = {
        "modelo": nombre,
        "hiperparametros": hiperparametros,
        "accuracy_train": accuracy_score(ytr, p_tr),
        "accuracy_test": accuracy_score(yte, p_te),
        "f1_macro_train": f1_score(ytr, p_tr, average="macro"),
        "f1_macro_test": f1_score(yte, p_te, average="macro"),
        "recall_optimo": rec[0],
        "recall_mantenimiento": rec[1],
        "recall_descarte": rec[2],
        "tiempo_entrenamiento_s": t_fit,
        "tiempo_prediccion_test_s": t_pred,
    }
    m["diagnostico"] = diagnostico(m["f1_macro_train"], m["f1_macro_test"])
    return m, p_te


def main(data_path: str, output_dir: str) -> None:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    # 1. Carga ---------------------------------------------------------------
    log("=== 1. CARGA DE DATOS ===")
    df = pd.read_csv(data_path)
    log(f"Registros cargados: {len(df):,} | Columnas: {df.shape[1]}")
    nulos = pp.verificar_nulos(df)
    log(f"Imputacion de nulos con la media: ya aplicada previamente "
        f"(verificacion: {nulos} NaN restantes)")

    # 2. Etiquetas y variables -----------------------------------------------
    log("=== 2. PREPARACION ===")
    X, y = pp.preparar_datos(df)
    log(f"Umbral de mantenimiento: vida restante < {pp.UMBRAL_MANTENIMIENTO} ciclos")
    log(f"Variables excluidas: {len(pp.columnas_a_excluir(df))} "
        f"(identificacion, *_score, vida restante, falla)")
    log(f"Variables de entrada: {X.shape[1]}")
    dist = y.map(pp.CLASES).value_counts()
    for k, v in dist.items():
        log(f"  Clase {k}: {v:,} ({v / len(y):.1%})")

    # 3. Split ---------------------------------------------------------------
    log("=== 3. DIVISION TRAIN / TEST (80/20 estratificada, seed=42) ===")
    X_tr, X_te, y_tr, y_te = pp.dividir(X, y)
    log(f"Train: {len(X_tr):,} registros | Test: {len(X_te):,} registros")

    pre = pp.construir_preprocesador(X_tr)
    Xtr = pre.fit_transform(X_tr)  # ajustado SOLO con Train (sin fuga)
    Xte = pre.transform(X_te)
    feature_names = list(pre.get_feature_names_out())
    log(f"Escalado + One-Hot ajustados con Train: {Xtr.shape[1]} features")

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=pp.SEED)
    resultados = []

    # 4. Baseline ------------------------------------------------------------
    log("=== 4. BASELINE: LogisticRegression (multinomial) ===")
    t0 = time.perf_counter()
    gs = GridSearchCV(
        LogisticRegression(max_iter=3000),
        {"C": [0.01, 0.1, 1, 10], "class_weight": [None, "balanced"]},
        scoring="f1_macro", cv=cv, n_jobs=-1, verbose=1,
    )
    gs.fit(Xtr, y_tr)
    base = gs.best_estimator_
    t_base = time.perf_counter() - t0
    log(f"Mejores parametros: {gs.best_params_} | F1 macro CV = {gs.best_score_:.4f}")
    m_base, p_base = evaluar("LogisticRegression (baseline)", base, Xtr, y_tr, Xte, y_te,
                             t_base, dict(gs.best_params_))
    resultados.append(m_base)
    log(f"Baseline -> Test Acc {m_base['accuracy_test']:.4f} | "
        f"F1 macro {m_base['f1_macro_test']:.4f}")

    # 5. Modelo principal ----------------------------------------------------
    log("=== 5. PRINCIPAL: XGBoost (busqueda aleatoria con CV estratificada) ===")
    t0 = time.perf_counter()
    espacio = {
        "max_depth": [3, 4, 5, 6],
        "learning_rate": [0.03, 0.05, 0.1],
        "n_estimators": [200, 300, 500],
        "subsample": [0.7, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "min_child_weight": [1, 3, 5, 10],
        "reg_lambda": [1, 5, 10],
        "gamma": [0, 0.5, 1],
        # 0 = sin ponderar, 0.5 = ponderacion suavizada (raiz), 1 = 'balanced' completo
        "peso_potencia": [0.0, 0.5, 1.0],
    }
    N_ITER = 15
    log(f"{N_ITER} combinaciones x 3 folds = {N_ITER * 3} entrenamientos (metrica: F1 macro sin ponderar)")
    y_np = y_tr.to_numpy()
    mejor_score, mejor_p = -1.0, None
    for i, p in enumerate(ParameterSampler(espacio, n_iter=N_ITER, random_state=pp.SEED), 1):
        hp = {k: v for k, v in p.items() if k != "peso_potencia"}
        scores = []
        for idx_tr, idx_va in cv.split(Xtr, y_np):
            w = pesos(y_np[idx_tr], p["peso_potencia"])
            m = XGBClassifier(objective="multi:softprob", num_class=3, tree_method="hist",
                              random_state=pp.SEED, n_jobs=-1, **hp)
            m.fit(Xtr[idx_tr], y_np[idx_tr], sample_weight=w)
            scores.append(f1_score(y_np[idx_va], m.predict(Xtr[idx_va]), average="macro"))
        sc = float(np.mean(scores))
        log(f"  [{i:>2}/{N_ITER}] F1 macro CV = {sc:.4f} | depth={p['max_depth']} lr={p['learning_rate']} "
            f"n={p['n_estimators']} peso^{p['peso_potencia']}")
        if sc > mejor_score:
            mejor_score, mejor_p = sc, p
    log(f"Mejores hiperparametros: {mejor_p}")
    log(f"F1 macro CV (3 folds) = {mejor_score:.4f}")
    potencia = mejor_p["peso_potencia"]
    mejor_hp = {k: v for k, v in mejor_p.items() if k != "peso_potencia"}

    log("Refit final con Early Stopping (10% de Train como validacion, paciencia=30)")
    X_a, X_v, y_a, y_v = train_test_split(
        Xtr, y_tr, test_size=0.10, stratify=y_tr, random_state=pp.SEED)
    sw_a = pesos(y_a.to_numpy(), potencia)
    sw_v = pesos(y_v.to_numpy(), potencia)
    params = {**mejor_hp, "n_estimators": 1000}
    xgb = XGBClassifier(
        objective="multi:softprob", num_class=3, tree_method="hist",
        eval_metric="mlogloss", random_state=pp.SEED, n_jobs=-1,
        early_stopping_rounds=30, **params,
    )
    xgb.fit(X_a, y_a, sample_weight=sw_a, eval_set=[(X_v, y_v)],
            sample_weight_eval_set=[sw_v], verbose=False)
    t_xgb = time.perf_counter() - t0
    best_it = int(xgb.best_iteration) + 1
    log(f"Early stopping: se detuvo en {best_it} arboles (de 1000 posibles)")
    hp_final = {**mejor_p, "n_estimators_efectivos": best_it, "early_stopping_rounds": 30}
    m_xgb, p_xgb = evaluar("XGBoost (principal)", xgb, Xtr, y_tr, Xte, y_te, t_xgb, hp_final)
    resultados.append(m_xgb)
    log(f"XGBoost -> Test Acc {m_xgb['accuracy_test']:.4f} | "
        f"F1 macro {m_xgb['f1_macro_test']:.4f}")

    # 6. Reporte final -------------------------------------------------------
    log("=== 6. REPORTE FINAL EN TEST SET ===")
    for nombre, pred in [("LogisticRegression (baseline)", p_base), ("XGBoost (principal)", p_xgb)]:
        print(f"\n--- {nombre} ---")
        print(classification_report(y_te, pred, labels=LABELS, target_names=NOMBRES, digits=4))
    for m in resultados:
        log(f"{m['modelo']}: Train F1 {m['f1_macro_train']:.4f} vs Test F1 "
            f"{m['f1_macro_test']:.4f} -> {m['diagnostico']}")

    # 7. Guardado de artefactos ---------------------------------------------
    log("=== 7. GUARDADO DE ARTEFACTOS ===")
    cat_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    num_cols = [c for c in X.columns if c not in cat_cols]
    metadata = {
        "clases": {str(k): v for k, v in pp.CLASES.items()},
        "umbral_mantenimiento": pp.UMBRAL_MANTENIMIENTO,
        "columnas_entrada": list(X.columns),
        "columnas_numericas": num_cols,
        "columnas_categoricas": cat_cols,
        "categorias": {c: sorted(X[c].unique().tolist()) for c in cat_cols},
        "features_transformadas": feature_names,
        "medias_entrenamiento": {c: float(X_tr[c].mean()) for c in num_cols},
    }
    joblib.dump(xgb, out / "modelo_baterias.joblib")
    joblib.dump(base, out / "modelo_baseline.joblib")
    joblib.dump(pre, out / "preprocessor.joblib")
    (out / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    (out / "metricas.json").write_text(json.dumps(resultados, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
    test_dir = Path(data_path).resolve().parent
    test_df = X_te.copy()
    test_df["estado_bateria"] = y_te
    test_df.to_csv(test_dir / "test_set.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, (nombre, pred) in zip(axes, [("Baseline: LogisticRegression", p_base), ("Principal: XGBoost", p_xgb)]):
        ConfusionMatrixDisplay.from_predictions(
            y_te, pred, labels=LABELS, display_labels=NOMBRES, ax=ax,
            cmap="Blues", colorbar=False, xticks_rotation=20)
        ax.set_title(nombre)
    fig.tight_layout()
    fig.savefig(out / "matriz_confusion.png", dpi=130)
    log(f"Artefactos guardados en: {out}/")
    log("Entrenamiento finalizado correctamente.")


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--data", required=True, help="CSV procesado (nulos ya imputados)")
    ap.add_argument("--output", default="models/", help="Carpeta de artefactos")
    args = ap.parse_args()
    main(args.data, args.output)

In [ ]:
%%writefile src/models/predict.py
"""Inferencia desacoplada con explicabilidad (SHAP).

Los artefactos (modelo, preprocesador, metadatos y explainer SHAP) se cargan en
memoria UNA sola vez (lru_cache); las llamadas posteriores a `predecir()` los
reutilizan.

Ejemplo:
    from src.models.predict import predecir
    resultado = predecir({...}) # dict o string JSON con las variables de la bateria
"""
from __future__ import annotations

import json
from functools import lru_cache
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import shap

MODELS_DIR = Path(__file__).resolve().parents[2] / "models"


@lru_cache(maxsize=1)
def cargar_artefactos(models_dir: str = str(MODELS_DIR)) -> dict:
    d = Path(models_dir)
    modelo = joblib.load(d / "modelo_baterias.joblib")
    return {
        "modelo": modelo,
        "pre": joblib.load(d / "preprocessor.joblib"),
        "meta": json.loads((d / "metadata.json").read_text(encoding="utf-8")),
        "explainer": shap.TreeExplainer(modelo),
    }


def _shap_de_clase(shap_values, clase: int) -> np.ndarray:
    """Normaliza la salida de SHAP (cambia segun la version de la libreria)."""
    if isinstance(shap_values, list):  # shap < 0.45: lista [clase] -> (n, f)
        return np.asarray(shap_values[clase])[0]
    arr = np.asarray(shap_values)  # shap >= 0.45: (n, f, clases)
    return arr[0, :, clase] if arr.ndim == 3 else arr[0]


def _valor_base(explainer, clase: int) -> float:
    ev = np.atleast_1d(explainer.expected_value)
    return float(ev[clase] if len(ev) > 1 else ev[0])


def predecir(muestra, top_n: int = 5, models_dir: str | None = None) -> dict:
    """Predice el estado de una bateria y explica el porque.

    Parametros
    ----------
    muestra : dict | str(JSON) con las variables de entrada de UNA bateria.
              Se ignoran claves extra (id, marca, etc.).
    top_n   : numero de variables mas influyentes a devolver.
    """
    art = cargar_artefactos(models_dir) if models_dir else cargar_artefactos()
    meta, pre, modelo, explainer = art["meta"], art["pre"], art["modelo"], art["explainer"]

    if isinstance(muestra, str):
        muestra = json.loads(muestra)
    if not isinstance(muestra, dict):
        raise TypeError("La muestra debe ser un dict o un string JSON.")

    cols = meta["columnas_entrada"]
    faltantes = [c for c in cols if c not in muestra]
    if faltantes:
        raise ValueError(f"Faltan {len(faltantes)} variables requeridas: {faltantes}")
    nulos = [c for c in cols if muestra[c] is None or (isinstance(muestra[c], float) and np.isnan(muestra[c]))]
    if nulos:
        raise ValueError(f"Valores nulos no permitidos en: {nulos}")

    fila = pd.DataFrame([{c: muestra[c] for c in cols}])
    Xt = pre.transform(fila)
    proba = modelo.predict_proba(Xt)[0]
    clase = int(np.argmax(proba))
    etiqueta = meta["clases"][str(clase)]

    # --- Explicabilidad: SHAP de la clase predicha, agregado por variable original
    sv = _shap_de_clase(explainer.shap_values(Xt), clase)
    cat_cols = meta["columnas_categoricas"]
    agregado = {}
    for nombre, valor in zip(meta["features_transformadas"], sv):
        origen = next((c for c in cat_cols if nombre.startswith(c + "_")), nombre)
        agregado[origen] = agregado.get(origen, 0.0) + float(valor)

    total_abs = sum(abs(v) for v in agregado.values()) or 1.0
    top = sorted(agregado.items(), key=lambda kv: abs(kv[1]), reverse=True)[:top_n]
    explicacion = []
    for var, contrib in top:
        item = {
            "variable": var,
            "valor": muestra[var],
            "contribucion_shap": round(contrib, 4),
            "impacto_relativo_pct": round(100 * abs(contrib) / total_abs, 1),
            "efecto": f"{'empuja hacia' if contrib > 0 else 'aleja de'} '{etiqueta}'",
        }
        if var in meta["medias_entrenamiento"]:
            item["promedio_flota"] = round(meta["medias_entrenamiento"][var], 3)
        explicacion.append(item)

    factores = "; ".join(
        f"{e['variable']}={e['valor']} ({e['efecto'].split(' ')[0]}, {e['impacto_relativo_pct']}%)"
        for e in explicacion[:3]
    )
    return {
        "clase_predicha": clase,
        "etiqueta": etiqueta,
        "confianza": round(float(proba[clase]), 4),
        "probabilidades": {meta["clases"][str(i)]: round(float(p), 4) for i, p in enumerate(proba)},
        "explicacion": explicacion,
        "valor_base_shap": round(_valor_base(explainer, clase), 4),
        "resumen": f"Bateria clasificada como '{etiqueta}' con {proba[clase]:.1%} de confianza. "
                   f"Factores principales: {factores}.",
    }